# Bölüm 1 — Keşifsel Veri Analizi ve İstatistik

**Proje:** GPT Eklenti Ekosisteminde Gizlilik Riski Analizi ve Otomatik Sınıflandırma

**Bu bölümde:**
- Veri setinin temel yapısını inceleyeceğiz (satır/sütun sayısı, eksik veriler, tipler)
- Kategori dağılımlarını görselleştireceğiz
- **Araştırma Sorusu 1:** GPT eklentilerinin istediği veriler kategorilere nasıl dağılıyor, hassas kategoriler toplamın ne kadarını oluşturuyor?
- **Araştırma Sorusu 2:** Hassas kategorilerde açıklama (description) yazılma oranı, hassas olmayanlara göre farklı mı? (istatistiksel test)

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

DATA_PATH = '../backend/data_entries_final.json'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
df.shape

(12811, 6)

## Adım 2 — Veri Yapısını İnceleme

Veriyi yükledik ama içeriğine henüz bakmadık. Bu adımda üç şeye bakıyoruz:

1. **Sütun tipleri** — `plugin_id_filenames` bir liste sütunu, diğerleri metin. Pandas bunları doğru okumuş mu kontrol ediyoruz.
2. **Eksik veriler** — özellikle `description` sütununda kaç kayıt boş bırakılmış? Bu, Araştırma Sorusu 2 için (hassas/hassas olmayan kategorilerde açıklama yazma oranı) referans noktamız olacak.
3. **Örnek satırlar** — verinin gerçekte nasıl göründüğünü gözle görmek, sayılara güvenmeden önce sağlama yapmak için.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12811 entries, 0 to 12810
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   index                12811 non-null  int64 
 1   plugin_id_filenames  12811 non-null  object
 2   name                 12811 non-null  str   
 3   description          12811 non-null  str   
 4   main_data_type       12811 non-null  str   
 5   data_type            12811 non-null  str   
dtypes: int64(1), object(1), str(4)
memory usage: 600.6+ KB


Dikkat: `description` gibi metin sütunlarında "eksik veri" genelde `NaN` değil, **boş string (`""`)** olarak saklanır. Bu yüzden sadece `.isna()` ile kontrol etmek yanıltıcı olur — boş string'leri de ayrıca sayıyoruz.

In [3]:
text_cols = ['name', 'description', 'main_data_type', 'data_type']

missing_summary = pd.DataFrame({
    'null_count': df[text_cols].isna().sum(),
    'empty_string_count': (df[text_cols] == '').sum(),
})
missing_summary['effectively_missing'] = missing_summary['null_count'] + missing_summary['empty_string_count']
missing_summary['pct_missing'] = (missing_summary['effectively_missing'] / len(df) * 100).round(2)
missing_summary

,null_count,empty_string_count,effectively_missing,pct_missing
name,0,1,1,0.01
description,0,1999,1999,15.60
main_data_type,0,0,0,0.00
data_type,0,0,0,0.00


In [4]:
df.sample(5, random_state=42)

,index,plugin_id_filenames,name,description,main_data_type,data_type
772,772,[g-2Eo3NxuS7_DX9y1883rIre0QLOgDFhKZit],files,List of files to include in the Repl.,Files and documents,File list
10196,10196,[g-nU9eHhPrc_gzm_cnf_XgzC7h8mvJI9f4uI9IAghnGS~...,chunks,Splits patterns or html into the amount of chu...,App usage data,Current session setting
385,385,"[g-009xFjOLC_OoDqO0qGP1tw2zGRYCYIfpUK, g-2UZly...",user_email,Email of the user. Has to be a real active ema...,Personal information,Email address
4269,4269,[g-GN0Zq0iZP_gzm_cnf_GpnRX0zniTgU24lIsjHtynV1~...,identifiers,"Filter by identifiers, comma-separated",Identifier,Resource IDs
8,8,[g-dnFnjifmN_gzm_cnf_JriSWBp4x1f0ITBun22ZIjjq~...,num_results,How many results to return. Defaults to 5. It ...,App usage data,Current session setting
